In [1]:
import pyspark
from pyspark.sql import SparkSession

# Create a local spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework_6") \
    .getOrCreate()

# Execute spark.version
print("Spark Version:", spark.version)

26/03/09 10:43:02 WARN Utils: Your hostname, shed resolves to a loopback address: 127.0.1.1; using 192.168.0.124 instead (on interface wlo1)
26/03/09 10:43:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 10:43:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.3


26/03/09 10:43:15 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
# Read the November 2025 Yellow data
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

# Repartition to 4 and save
df.repartition(4).write.parquet("hw_q2_output", mode="overwrite")

In [3]:
from pyspark.sql import functions as F

# Load the zone lookup data
df_zones = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")

# Create Temp Views
df.createOrReplaceTempView("yellow_data")
df_zones.createOrReplaceTempView("zones")

**Question 3**

In [4]:
spark.sql("""
    SELECT COUNT(1) AS total_trips
    FROM yellow_data
    WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

+-----------+
|total_trips|
+-----------+
|     162604|
+-----------+



**Question 4**

In [6]:
spark.sql("""
    SELECT 
        MAX(unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS max_duration_hours
    FROM yellow_data
""").show()

+------------------+
|max_duration_hours|
+------------------+
| 90.64666666666666|
+------------------+



**Question 6**

In [7]:
spark.sql("""
    SELECT z.Zone, COUNT(1) as trip_count
    FROM yellow_data y
    JOIN zones z ON y.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY trip_count ASC
    LIMIT 5
""").show(truncate=False)

[Stage 12:===========================================>              (6 + 2) / 8]

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Governor's Island/Ellis Island/Liberty Island|1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Arden Heights                                |1         |
|Port Richmond                                |3         |
|Rikers Island                                |4         |
+---------------------------------------------+----------+

